<a href="https://colab.research.google.com/github/chandupradeep18/Building-with-the-Claude-API/blob/main/exercise_prompt_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install anthropic

In [29]:
from anthropic import Anthropic
from google.colab import userdata
import json
from datetime import  datetime
from statistics import mean
import re
import ast
import os
import csv

In [30]:
client=Anthropic(api_key=userdata.get('Claude'))
model='claude-haiku-4-5-20251001'
max_tokens=1000

In [31]:
def add_user_message(messages,text):
  messages.append({"role":"user","content":text})
def add_assistant_message(messages,text):
  messages.append({"role":"assistant","content":text})
def chat(message,system=None,temperature=1.0,stop_sequences=[]):
  param={
      'model':model,
      'max_tokens':max_tokens,
      'temperature':temperature,
      'messages':message
  }
  if system:
    param['system']=system
  if stop_sequences:
    param["stop_sequences"] = stop_sequences
  response=client.messages.create(**param)
  details={'Time':datetime.now().isoformat(),'id':response.id,'role':response.role,'stop_reason':response.stop_reason,'input_tokens':response.usage.input_tokens,'output_tokens':response.usage.output_tokens,'user':message,'assistant':response.content[0].text}
  Collect_Response(details)
  return response.content[0].text
def Collect_Response(details):
  file_exist=os.path.exists('/content/drive/MyDrive/ClaudePromptEval/prompt_eval_response_log.csv')
  with open('/content/drive/MyDrive/ClaudePromptEval/prompt_eval_response_log.csv','a',newline='') as file:
    writer=csv.DictWriter(file,fieldnames=details.keys())
    if not file_exist:
      writer.writeheader()
    writer.writerow(details)

In [46]:
#draft A Prompt and create a Dataset
def generate_dataset():
  prompt="""
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

  Example output:
  ```json
  [
    {
       "task": "Description of task",
       "format":"python" or "json" or
    },
    ...additional
  ]
  ```

  * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
  * Focus on tasks that do not require writing much code

  Please generate 3 objects."""
  messages=[]
  add_user_message(messages,prompt)
  add_assistant_message(messages,"```json")
  text=chat(messages,stop_sequences=["```"])
  with open('eval_dataset.json','w') as file:
    json.dump(json.loads(text),file,indent=2)

In [47]:
generate_dataset()

In [58]:
#feed to grader
def grade_by_model(test_case, output):
  eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

  messages=[]
  add_user_message(messages,eval_prompt)
  add_assistant_message(messages,"```json")
  text=chat(messages,stop_sequences=["```"])
  return json.loads(text)

In [59]:
#feed through Claude
def run_prompt(test_case):
  prompt = f"""
  Please solve the following task in max token of 500 only:
  {test_case['task']}

  * Respond only with Python, JSON, or a plain Regex
  * Do not add any comments or commentary or explanation
  """
  messages=[]
  add_user_message(messages,prompt)
  add_assistant_message(messages,"```code")
  text=chat(messages,stop_sequences=["```"])
  return text

def run_test_cases(test_case):
  output=run_prompt(test_case)
  score=grade_by_model(test_case,output)
  return {
      'input':test_case,
      'output':output,
      'score':score
  }

def run_eval(dataset):
  results=[]
  for data in dataset:
    results.append(run_test_cases(data))
  return results

In [60]:
with open('eval_dataset.json','r') as file:
  dataset=json.load(file)
result=run_eval(dataset)
with open('eval_result.json','w') as file:
  json.dump(result,file,indent=2)